# Weekly Planning: starter notebook for Data Analysts

Connects to the mission database through an **SSH tunnel** and runs a few example queries. It holds **no secrets**: you type your password when asked, and it is never saved in this file.

**Before you start**, open the tunnel in a terminal and leave it open (see `docs/DBEAVER.md`):

```
ssh -L 54322:127.0.0.1:54322 <your-server-user>@<the-server>
```

Your login is personal (made by `tools/add-analyst-login.sh`). It can read, write and create in the schemas `public`, `portal` and `dashboards` and sees every row. **Changes are live data.** This notebook opens a *read-only* session by default; to write, change `READ_ONLY` below on purpose.

In [ ]:
# Once per computer:  pip install pandas sqlalchemy psycopg2-binary
import os
from getpass import getpass

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 160)

# The tunnel makes the server's database appear on this computer. Nothing here is a secret.
HOST = os.environ.get('GFM_DB_HOST', '127.0.0.1')
PORT = int(os.environ.get('GFM_DB_PORT', '54322'))
DATABASE = os.environ.get('GFM_DB_NAME', 'postgres')
USER = os.environ.get('GFM_DB_USER') or input('Your database login: ')
READ_ONLY = True   # set to False only when you mean to change data; try risky SQL inside BEGIN ... ROLLBACK first

# The password is asked for here (or taken from GFM_DB_PASSWORD in your own shell). It is never written into this notebook.
PASSWORD = os.environ.get('GFM_DB_PASSWORD') or getpass('Password (not shown): ')

engine = create_engine(
    URL.create('postgresql+psycopg2', username=USER, password=PASSWORD, host=HOST, port=PORT, database=DATABASE),
    connect_args={'options': '-c default_transaction_read_only=' + ('on' if READ_ONLY else 'off')},
    pool_pre_ping=True,
)
del PASSWORD

def q(sql, **params):
    """Run a SELECT and return a DataFrame. q('select * from public.missions where id = :id', id=1)"""
    with engine.connect() as conn:
        return pd.read_sql(text(sql), conn, params=params)

q('select current_user as login, current_database() as db, now() as server_time')

## 1. What is there?
The three schemas you work in: `public` (planning data), `portal` (portal settings) and `dashboards` (the views behind the Dashboards).

In [ ]:
q("""
select table_schema, table_type, count(*) as objects
from information_schema.tables
where table_schema in ('public', 'portal', 'dashboards')
group by 1, 2 order by 1, 2
""")

In [ ]:
# The views behind the Dashboards
views = q("""
select table_name as view_name
from information_schema.views
where table_schema = 'dashboards' order by 1
""")
views

In [ ]:
# The first rows of the first dashboards view (change the name to look at another)
if len(views):
    first = views.iloc[0]['view_name']
    print('dashboards.' + first)
    display(q(f'select * from dashboards."{first}" limit 10'))

## 2. Example queries

In [ ]:
# The mission(s) in this database
q('select id, name, default_language, time_zone from public.missions order by id')

In [ ]:
# How many answers each Weekly Planning question has, most answered first
q("""
select question_key, count(*) as answers
from public.weekly_planning_answers
group by 1 order by 2 desc limit 20
""")

In [ ]:
# The biggest tables (a quick idea of where the data is)
q("""
select schemaname as schema, relname as table_name, n_live_tup as approx_rows
from pg_stat_user_tables
where schemaname in ('public', 'portal')
order by n_live_tup desc limit 15
""")

## 3. Draw something
Any query result is a DataFrame, so it can be charted: `df.plot(...)`, or export it with `df.to_csv('my-result.csv', index=False)` (keep exports with people's names out of Git).

In [ ]:
df = q("""
select question_key, count(*) as answers
from public.weekly_planning_answers
group by 1 order by 2 desc limit 10
""")
if len(df):
    df.set_index('question_key')['answers'].plot(kind='barh', figsize=(8, 4), title='Most answered questions')

## 4. Writing (only on purpose)
Set `READ_ONLY = False` in the first cell and run it again. Always try a change inside a transaction first and look at the result before you commit:

```python
with engine.begin() as conn:                       # begin() commits at the end; raise an error to roll back
    conn.execute(text('update public.some_table set x = 1 where id = :id'), {'id': 1})
    print(conn.execute(text('select * from public.some_table where id = :id'), {'id': 1}).fetchall())
    raise RuntimeError('rolled back on purpose')   # remove this line when you are sure
```

The nightly backup is the way back for a mistake that was committed.